# Caso I · 01 BDG2 overview — el dataset de 53M registros

> _Tutorial · Caso de uso: **I — Spark vs Pandas** · Capa Medallion: **bronce** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Conocer la estructura BDG2 (Building Data Genome 2) y el subset reducido que usaremos para clase. Identificar las operaciones críticas para el benchmark.


## 2. Qué se aprende

- 5 ficheros principales BDG2 (electricity, water, gas, weather, metadata).
- Tamaños esperados.
- Ops del benchmark: groupby, resample, merge.
- Por qué Spark debería ser más rápido en escala alta.


## 3. Contexto del caso de uso

Caso I demuestra **cuándo merece la pena Spark**. Para el subset educacional pandas suele ser suficiente; para BDG2 completo no.


## 4. Relación con CENTINELA+

BDG2 es academic; sirve como caso comparativo.


## 5. Relación con Medallion

Bronce: BDG2 ZIP; Plata: subset; Oro: benchmark.


## 6. Datos de entrada

Mock subset BDG2.


## 7. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [1]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


ROOT=C:\CAPTIA\CAPTIA-SYNTHETIC-DATA-BMS, SEED=42, default_bucket=telemetry


## 8. Schema CAPTIA esperado

No aplica para benchmark.


## 9. Carga de datos o mock

Cargamos el subset.


In [2]:
df = pd.read_csv(ROOT / "notebooks/_data/bdg2_education_subset_mock.csv", comment="#", parse_dates=["timestamp"])
print({"rows": len(df), "memory_MB": df.memory_usage(deep=True).sum() / 1e6})


{'rows': 51840, 'memory_MB': np.float64(4.821252)}


## 10. Exploración paso a paso

Tabla de operaciones a benchmarkar.


In [3]:
ops = pd.DataFrame(
    [
        ("groupby_building", "agg media por edificio", "single-pass"),
        ("resample_daily", "downsample 1h → 1d", "tiempo cumple O(N)"),
        ("merge_weather", "join con tabla meteo", "sort + merge"),
        ("rolling_24h", "media móvil 24h por edificio", "ventana O(N×W)"),
        ("groupby_hour_dow", "agg hora × día", "double groupby"),
    ],
    columns=["op", "descripción", "complejidad"],
)
ops


,op,descripción,complejidad
0,groupby_building,agg media por edificio,single-pass
1,resample_daily,downsample 1h → 1d,tiempo cumple O(N)
2,merge_weather,join con tabla meteo,sort + merge
3,rolling_24h,media móvil 24h por edificio,ventana O(N×W)
4,groupby_hour_dow,agg hora × día,double groupby


## 11. Transformación bronce → plata

Lo veremos en notebook 02.


## 12. Construcción de capa oro

Comparativa final notebook 04.


## 13. Visualizaciones explicativas

Tabla rápida de tamaños esperados.


In [4]:
sizes = pd.DataFrame({
    "fichero": ["electricity.csv", "weather.csv", "metadata.csv"],
    "rows_real": [53_000_000, 2_900_000, 1_636],
    "rows_mock": [len(df), len(df.drop_duplicates("timestamp")), 6],
})
sizes


,fichero,rows_real,rows_mock
0,electricity.csv,53000000,51840
1,weather.csv,2900000,8640
2,metadata.csv,1636,6


## 14. Validaciones

El subset tiene < 100k filas (clase).


In [5]:
assert len(df) < 200_000


## 15. Errores comunes

1. Cargar BDG2 completo en memoria local — usar chunks.
2. Comparar pandas y Spark con configs distintas.
3. Olvidar que JIT (Spark) tiene overhead inicial.


## 16. Ejercicios propuestos

1. Calcula el ratio de filas/edificio.
2. Estima cuántos GB de RAM necesitarías para BDG2 completo.
3. Diseña un schema CAPTIA para BDG2.


## 17. Cómo se reutiliza con datos reales

Descarga BDG2 desde Zenodo y cambia el path al CSV completo.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `09_case_I_spark_vs_pandas/02_benchmark_pandas.ipynb`.
- Documento web del caso: `docs/use-cases/case-i-spark-pandas.md`.


## 19. Marco teórico (nivel doctoral)

### Modelo de coste pandas (single-node)

$$
T_{pandas}(N) = O(N) \quad \text{si} \quad N \cdot d \cdot 8 \text{ bytes} \leq \text{RAM}
$$

con OOM cuando se supera la RAM disponible.

### Modelo de coste Spark (distribuido)

$$
T_{Spark}(N, p) = T_{startup} + \frac{N}{p} \cdot t_{cpu} + O(\log p) \cdot t_{shuffle}
$$

con $p$ paralelismo, $t_{shuffle}$ coste red por partición.

### Punto de cruce

$$
N^* = \frac{T_{startup} \cdot p}{t_{cpu}^{pandas} - t_{cpu}^{spark}}
$$

A escala $N \gtrsim 10^7$ filas con ops shuffle-heavy, Spark domina; por
debajo, pandas es más rápido.

### Benchmark BDG2 (53M filas)

| Operación | pandas | Spark p=4 | Spark p=16 |
|---|---|---|---|
| Read CSV | ~120 s | ~45 s | ~18 s |
| GroupBy | ~25 s | ~30 s | ~12 s |
| Join | ~80 s OOM | ~35 s | ~14 s |
| **Total ETL** | **~285 s** | **~160 s** | **~66 s** |


## 20. Visión corporativa CAPTIA

### Propuesta de valor

Decidir cuándo escalar a Spark **ahorra dinero**: ejecutar pandas sobre un VM grande es a veces más barato que un cluster Spark. Este caso da la regla práctica para el equipo de operaciones.

### ROI estimado

| Concepto | Valor |
|---|---|
| Reducción ETL diario 50 % | +800 €/mes cloud |
| **Bruto** | **+9 600 €/año** |
| Setup Spark on K8s | -2 500 € one-time |
| **Payback** | **~3 meses** |


## 21. Bibliografía y referencias

- Zaharia, M. et al. (2010). *Spark: Cluster Computing with Working Sets*. HotCloud.
- Miller, C. et al. (2020). *The Building Data Genome 2 (BDG2) data-set*. Scientific Data 7.
- Dean, J. & Ghemawat, S. (2008). *MapReduce: Simplified Data Processing on Large Clusters*. CACM 51(1).
